# LSTMm data generation

Long Quach 

Apr

In this version, we assume that the FED has the most up-to-date data when they make the decision

In [ ]:
import os
from datetime import date, datetime
from functools import partial

import copy
import math
import random
import timeit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.api import VAR
from tqdm import tqdm

from sklearn.model_selection import KFold, StratifiedKFold, TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


## Functions:


##Cross-validation section
# Auxiliary functions used to estimate CV mean loss
# constructDataLoader: creates train and test data loaders. The batch size is set equal to the length of the test set to avoid.
# train_test: evaluates of the loss un the test dataset in a single fold.
# timeseriesCV: finds mean test loss across all folds

# constructDataLoader()

def constructDataLoader(split, X, y, shuffle= True):
    train_idx = split[0]
    test_idx = split[1]
    batch_size = len(test_idx) # different from user input

    # train dataset
    features = X[train_idx,:,:]
    labels = y[train_idx,:,:]
    dataset = TensorDataset(features, labels)
    train_data = DataLoader(dataset, batch_size = batch_size, shuffle = shuffle)

    # test dataset
    features = X[test_idx,:,:]
    labels = y[test_idx,:,:]
    dataset = TensorDataset(features, labels)
    test_data = DataLoader(dataset, batch_size = batch_size, shuffle = shuffle)
    return train_data, test_data


def train_test(train_data, test_data, num_epochs, device, criterion, model, optimizer):

    for layer in model.children():
        if hasattr(layer, 'reset_parameters'):
            layer.reset_parameters()

    epoch_train_loss = []
    epoch_test_loss = []

    model.train()

    for epoch in range(num_epochs):
        # Training phase
        train_losses = []
        for seq_batch, labels_batch in train_data:
            seq_batch = seq_batch.to(device)
            labels_batch = labels_batch.to(device)
            
            # # EXPLICITLY Reset hidden state
            h = None
            optimizer.zero_grad()                           

            # Forward pass
            y_pred, h = model(seq_batch, h)
            # y_pred, _ = model(seq_batch)

            loss = criterion(y_pred, labels_batch)
            
            # backward pass
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())        
        train_loss = sum(train_losses)/len(train_losses)
        epoch_train_loss.append(train_loss)
        
        # Add a testing phase
        model.eval()
        test_losses = []
        with torch.no_grad():
            for seq_batch, labels_batch in test_data:
                seq_batch = seq_batch.to(device)
                labels_batch = labels_batch.to(device)                
                # when using MSE_loss()
                # the predicted model has dimensions [batch_size, n_features]
                # The target, labels_batch, has dimensions [batch_size, output_size, n_features]            
                # reduce it to [batch_size, n_features
                y_pred, _ = model(seq_batch)            
                # if labels_batch.dim() == 3:
                #     labels_batch = labels_batch[:, -1, :]  # Use last time step
                loss = criterion(y_pred, labels_batch)
                test_losses.append(loss.item())
                
        test_loss  = sum(test_losses)/len(test_losses)
        epoch_test_loss.append(test_loss)
    
        if epoch % 200 == 0:
            print("Epoch: %d, train loss: %1.5f" % (epoch, loss.item())) 
            print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}')
        model.train()
    return epoch_test_loss[-1]


## Generate Timestep

##### XY_gen (train,n_input,n_output,overlapping = True)
##### Creates XX with n_input lags and YY with n_output lags. Set `overlapping` to make X_t and X_t+1 overlapping or not.
def XY_gen(train, n_input, n_output=1, X_input=[], Y_output=[], 
           exclude_y = True,
           overlapping=True, allowed_months=None):
    """
    Generate input-output sequences from a DataFrame, with optional filtering by output month.

    Parameters:
    - train (pd.DataFrame): Input data indexed by Date.
    - n_input (int): Number of time steps for input sequence.
    - n_output (int): Number of time steps for output sequence.
    - X_input (list): List of variable names to include in X.
    - Y_output (list): List of variable names to include in y.
    - overlapping (bool): Whether to allow overlapping sequences.
    - allowed_months (list): List of months (int) to keep based on y's first month.

    Returns:
    - X, y (list of np.array): Filtered lists of input and output sequences.
    """

    if not isinstance(train, pd.DataFrame):
        raise TypeError("'train' must be a pandas DataFrame")
    if not isinstance(train.index, pd.DatetimeIndex):
        raise ValueError("'train' must have a DatetimeIndex")

    # ---------- column selection ----------
    X_cols = list(train.columns) if not X_input else list(X_input)
    y_cols = list(train.columns) if not Y_output else list(Y_output)

    if exclude_y:
        X_cols = [c for c in X_cols if c not in y_cols]

    overlap = set(X_cols).intersection(y_cols)
    if overlap:
        raise ValueError(
            f"X and Y share columns {sorted(overlap)} – "
            "set exclude_y=True or adjust X_input / Y_output."
        )

    if len(X_cols) == 0:
        raise ValueError("X has no columns after exclusions.")

    # ---------- prepare arrays ----------
    train_X = train[X_cols].values
    train_Y = train[y_cols].values
    dates   = train.index

    X, y = [], []
    stride     = 1 if overlapping else n_input
    max_start  = len(train) - n_input - (0 if n_output == 0 else n_output)

    for in_start in range(0, max_start + 1, stride):
        in_end = in_start + n_input

        # choose y indices
        if n_output == 0:
            y_start, y_end = in_end - 1, in_end      # contemporaneous
        else:
            y_start, y_end = in_end, in_end + n_output

        # month filter
        if allowed_months and dates[y_start].month not in allowed_months:
            continue

        x_seq, y_seq = train_X[in_start:in_end], train_Y[y_start:y_end]
        if pd.isnull(x_seq).any() or pd.isnull(y_seq).any():
            continue

        X.append(x_seq)
        y.append(y_seq)

    return X, y

def timeSeriesCV(cv_splits, X, y, num_epochs, device, criterion, model, learn_rate):
    mean_loss = 0.0
    for i, split in enumerate(cv_splits):
        train_data, test_data = constructDataLoader(split, X, y)
        optimizer = torch.optim.Adam(model.parameters(), lr=learn_rate)                           
        test_loss  = train_test(train_data, test_data, num_epochs, 
                                device, criterion, model, optimizer)
        mean_loss += test_loss
        print(f'---------------------------------------------------')        
        print(f'Split: {i} - test_loss = {test_loss}')
        print(f'---------------------------------------------------')
    return mean_loss/len(cv_splits)    


# Define the evaluation function
def evaluate_config(input_data, model_class,criterion, n_splits, device, config, X_input=[], Y_output=['FF']):
    """Evaluate a single configuration using timeSeriesCV."""
    print(f"Evaluating configuration: {config}")

    # print("Inside evaluate_config, input_data type:", type(input_data))

    # Generate lagged data
    XX, YY = XY_gen(input_data, config['lag_num'], 1, X_input=X_input, Y_output=Y_output, overlapping=True)
    XX, YY = torch.tensor(np.array(XX)), torch.tensor(np.array(YY))

    # Split into train (CV) and test
    last_obs = round(XX.shape[0] * 0.8)
    X_cv = XX[:last_obs, :, :].float()
    y_cv = YY[:last_obs, :, :].float()

    max_splits = min(n_splits, (y_cv.shape[0] - config['lag_num']) // 2)  # Ensure enough data for splits
    test_size = max(1, (y_cv.shape[0] - config['lag_num']) // (max_splits + 1))  # Avoid zero-sized test sets
    
    if max_splits < 1:
        raise ValueError(
            f"Insufficient data for cross-validation: lag_num={config['lag_num']}, y_cv.shape[0]={y_cv.shape[0]}"
        )

    print(f"max_splits: {max_splits}")

    tscv = TimeSeriesSplit(n_splits=max_splits, test_size=test_size, gap=0)
    cv_splits = list(tscv.split(X_cv))

    # Define the model
    model = model_class(
        input_size=X_cv.shape[2],
        hidden_size=int(config['hidden_size']),
        output_size=y_cv.shape[2],
        num_layers=int(config['num_layers']),
        dropout=config['dropout']
    )
    
    model.reset_parameters()
    model.apply(init_weights)
    model.to(device)

    # Perform cross-validation
    mean_loss = timeSeriesCV(
        cv_splits=cv_splits,
        X=X_cv,
        y=y_cv,
        num_epochs=config['num_epochs'],
        device=device,
        criterion=criterion,
        model=model,
        learn_rate=config['learn_rate']
    )

    return mean_loss

# Predefine a space in teh functions file


def objective(params, input_data, model_class,criterion, n_splits, device, n_repeats=10, seed = 420, X_input=[], Y_output=['FF']):
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Ensure deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Map the list of parameters to their names
    config = {
        "lag_num": params[0],
        "learn_rate": round(params[1], 4),
        "num_epochs": params[2],
        "hidden_size": params[3],
        "num_layers": params[4],
        "dropout": round(params[5], 2)
    }

    losses = []
    # seeds = []
    for _ in range(n_repeats):
        # print("Type of input_data in objective:", type(input_data))

        loss = evaluate_config(input_data, model_class, criterion, n_splits, device, config, X_input=X_input, Y_output=Y_output)
        losses.append(loss)
        # seeds.append(seed)

    mean_loss = np.mean(losses)
    variance = np.var(losses)

    return mean_loss, variance #, seeds

def bayesian_optimization(input_data, model_class,criterion, n_splits, device, space,
 n_calls, n_initial_points= 10, n_repeats = 10, seed = 420, X_input=[], Y_output=['FF']):
    results = []
    
    def objective_partial(params):
        mean_loss, variance = objective(
            params, input_data, model_class, criterion, n_splits, device, n_repeats, seed, X_input, Y_output
        )

        # Store results for later analysis
        results.append({
            "config": {
                "lag_num": params[0],
                "learn_rate": round(params[1], 4),
                "num_epochs": params[2],
                "hidden_size": params[3],
                "num_layers": params[4],
                "dropout": round(params[5], 2)
            },
            "mean_loss": mean_loss,
            "variance": variance
            # "seeds": seeds
        })

        return mean_loss  # 

    # Run Bayesian Optimization
    configs = gp_minimize(
        objective_partial,
        space,
        n_calls=n_calls,
        n_initial_points=n_initial_points,
        initial_point_generator = "random",
        random_state=seed
        
    )

    # Find the best configuration based on variance (optimization target)
    best_index = np.argmin(configs.func_vals)

    # Extract the best configuration details
    best_result = results[best_index]
    best_config = best_result["config"]
    best_config["mean_loss"] = best_result["mean_loss"]
    best_config["variance"] = best_result["variance"]

    return best_config, results



def dataset_scaler(data, scaler_type="scale", min_val=0.05, delta=0.05, plot=False):
    """
    This function scales a dataset using different methods and returns the scaled data and the scaler object.

    Parameters:
    data: a pandas dataframe containing the original data
    scaler_type: a string indicating the type of scaling method to use ("scale", "minmax" or "noscale")
    min_val: a float indicating the minimum absolute value to keep in the scaled data
    delta: a float indicating the amount to add to values that are less than min_val

    Returns:
    scaled_data: a pandas dataframe containing the scaled data
    scaler: a scaler object that can be used for inverse transformation or applying to new data
    """

    # Choose scaler
    if scaler_type == "scale":
        scaler = StandardScaler()
    elif scaler_type == "minmax":
        scaler = MinMaxScaler(feature_range=(delta, 1))
    elif scaler_type == "noscale":
        scaler = None
    else:
        raise ValueError("Invalid scaler type")

    # Apply scaler and transform data
    if scaler is not None:
        scaler.fit(data)
        transformed_data = scaler.transform(data)
        np.putmask(transformed_data, abs(transformed_data) < min_val, transformed_data + delta * np.sign(transformed_data))
        # transformed_data = transformed_data[1:, :] # remove first row containing NaNs
    else:
        transformed_data = data.to_numpy()
        # transformed_data = transformed_data[1:, :]

    # Create pandas dataframe and plot data to check if transform is OK
    scaled_data = pd.DataFrame(transformed_data, index=data.index)
    # scaled_data = scaled_data.iloc[1:, :]

    if plot:
        scaled_data.plot()

    return scaled_data, scaler


def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight_ih' in name:
            torch.nn.init.xavier_uniform_(param.data)
        elif 'weight_hh' in name:
            torch.nn.init.xavier_uniform_(param.data)
        elif 'bias' in name:
            param.data.fill_(0)

def Train_DL_Model(model_class, XX,YY, n_features, hidden_size, num_layers, drop_out, learn_rate, num_epochs, criterion, device, print_log = False):

    start_time = timeit.default_timer()

    features = XX.float()
    labels = YY.float()
    dataset = TensorDataset(features, labels)
    batch_size = 64
    batch_size = min(batch_size, len(dataset))
    
    model_data = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    if num_layers == 1:
        drop_out = 0  # No dropout for single-layer models
        
    trained_model = model_class(input_size=n_features, hidden_size=hidden_size, 
                             output_size=YY.shape[2], num_layers=num_layers, 
                             dropout=drop_out)
    trained_model.train()
    trained_model.reset_parameters()
    trained_model.apply(init_weights) 
    trained_model.to(device)

    optimizer = torch.optim.Adam(trained_model.parameters(), lr=learn_rate)      

    epoch_train_loss = []
    for epoch in range(num_epochs):
        train_losses = []
        for seq_batch, labels_batch in model_data:
            optimizer.zero_grad()                    
            seq_batch = seq_batch.to(device)
            labels_batch = labels_batch.to(device)
            
            y_pred, _ = trained_model(seq_batch)
#             if y_pred.dim() == 3:  # If model outputs a sequence
#                 y_pred = y_pred[:, -1, :]  # Take the last time step: (batch_size, n_features)
            
#             # Ensure labels_batch matches y_pred
#             labels_batch = labels_batch.squeeze(1)  # Shape: (batch_size, n_features)
            
            loss = criterion(y_pred, labels_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())     
            
        train_loss = sum(train_losses)/len(train_losses)
        epoch_train_loss.append(train_loss)
        if epoch % 100 == 0:
            if print_log:
                print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f}')

    LSTMm_elapsed = timeit.default_timer() - start_time
    if print_log:
        print(f"Training completed in {LSTMm_elapsed:.2f} seconds")


    return trained_model, epoch_train_loss  

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#set seed just for the sake of reproducibility
### Get the seed from the file
seed = 2025
torch.manual_seed(seed)  # For CPU
torch.cuda.manual_seed(seed)  # For GPU (if using CUDA)
# torch.cuda.manual_seed_all(32025)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# LSTM
# LSTM

class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, dropout):
        super(LSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc2 = nn.Linear(hidden_size // 2, output_size)

    def forward(self, input_seq, h=None):
        lstm_out, h = self.lstm(input_seq, h)  # lstm_out: [batch, seq_len, hidden]
        x = lstm_out[:, -1, :]                 # Take last time step
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        predictions = self.fc2(x).unsqueeze(1) # Shape: [batch, 1, output_size]
        return predictions, h

    def reset_parameters(self):
        self.lstm.reset_parameters()
        self.fc1.reset_parameters()
        self.fc2.reset_parameters()
       
# GRU
    


def fixed_sample_experiment(cutoff_date, save_tag=None, save_csv=True, save_png=False):
    """
    Train an LSTM on data up to `cutoff_date`, predict forward, plot, and save.

    Reads notebook globals: scaled_data, scaler, lag_num, hidden_size, num_layers,
    drop_out, learn_rate, num_epochs, n_features, criterion, device, LSTM,
    XY_gen, Train_DL_Model, unscale_target_only.

    Args:
        cutoff_date: e.g. '2016-01-01' - first test date is 2 months later.
        save_tag: filename tag; defaults to the cutoff year.
        save_csv: write predictions to a timestamped CSV.
        save_png: also save the plot as a timestamped PNG.

    Returns:
        (actual_unscaled, pred_unscaled, ref_dates)
    """
    from datetime import datetime as _dt
    save_tag = save_tag or cutoff_date[:4]

    rolling_df = scaled_data.copy()
    cut_off_index = rolling_df.index.get_loc(cutoff_date)
    cut_off_index_offset = cut_off_index - lag_num
    ref_dates = pd.to_datetime(
        rolling_df.index[lag_num + cut_off_index_offset:], format='%Y-%d-%m'
    )

    all_XX, all_YY = XY_gen(
        rolling_df, lag_num, 1, X_input=[], Y_output=['FF'], overlapping=True
    )
    all_XX = torch.tensor(np.array(all_XX))
    all_YY = torch.tensor(np.array(all_YY))

    train_XX = all_XX[:cut_off_index_offset, :, :]
    train_YY = all_YY[:cut_off_index_offset, :, :]
    test_XX  = all_XX[cut_off_index_offset:, :, :]
    test_YY  = all_YY[cut_off_index_offset:, :, :]
    test_Y   = test_YY[:, 0, :]

    model, _ = Train_DL_Model(
        LSTM, train_XX, train_YY,
        int(n_features), int(hidden_size), int(num_layers), drop_out,
        learn_rate, num_epochs, criterion, device,
    )
    model.eval()
    with torch.no_grad():
        oos_pred, _ = model(test_XX.float().to(device))
    oos_pred = np.array(oos_pred.cpu())

    actual_unscaled, pred_unscaled = unscale_target_only(
        test_Y, oos_pred, scaled_data, scaler, target_col_index=0
    )

    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(15, 7))
    ax.plot(ref_dates[1:], actual_unscaled, linestyle='--', color='black', label='Actual')
    ax.plot(ref_dates[1:], pred_unscaled,  linestyle='-',  color='blue',
            label='Out-of-sample LSTM predictions')
    ax.set_ylabel('Fed Funds Rate')
    ax.set_xlabel('Date')
    ax.legend(loc='upper left')
    plt.tight_layout(rect=[0, 0, 1, 0.97])

    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    if save_png:
        plt.savefig(f'v3FixedSample{save_tag}_LSTMfoosFed_{ts}.png', dpi=300)
    plt.show()

    if save_csv:
        pd.DataFrame(pred_unscaled).to_csv(
            f'v3FixedSample{save_tag}_LSTM_unscaled_{ts}.csv', index=False
        )

    return actual_unscaled, pred_unscaled, ref_dates


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
seed = 2025
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---- Read and transform data ----
df_raw = pd.read_csv('Input/Data_LQ.csv')
df_raw = df_raw.set_index('Date')
df_raw.index = pd.to_datetime(df_raw.index, format="%d/%m/%Y")
df = df_raw.copy()

# Year-over-year transforms
df['MLU6yoy']  = df['MLU6'].diff(periods=12)                  # YoY change in employment
df['FM2yoy']   = df['FM2'].pct_change(periods=12) * 100       # YoY change in M2
df['LH']       = df['LH'].pct_change(periods=12) * 100        # YoY change in "not in workforce"
df['PCU']      = df['PCU'].pct_change(periods=12) * 100       # YoY PCE inflation
df['PCUSLFE']  = df['PCUSLFE'].pct_change(periods=12) * 100   # YoY core PCE inflation
df['JCBM']     = df['JCBM'].pct_change(periods=12) * 100
df['JCXFEBM']  = df['JCXFEBM'].pct_change(periods=12) * 100
df['USPHPIM']  = df['USPHPIM'].pct_change(periods=12) * 100
df['GSACPPIC'] = df['GSACPPIC'].pct_change(periods=12) * 100
df['PZRAW']    = df['PZRAW'].pct_change(periods=12) * 100
df['rawLKPRIVA'] = df['LKPRIVA']
df['LKPRIVA']  = df['LKPRIVA'].pct_change(periods=12) * 100
df['MGDPN']    = df['MGDPN'].pct_change(periods=12) * 100

# Yield-curve spreads
df['YC_2y_10y'] = df['Yield10Y'] - df['Yield2Y']
df['YC_6m_1y']  = df['Yield1Y']  - df['Yield6m']

# Target: shift FF by -1 so the model predicts next month's rate
df['FF'] = df['FF'].shift(-1)

# Drop columns we don't feed to the model
df.drop(['LANAGRD', 'LEPRIVA', 'Yield1Y', 'Yield10Y'], axis=1, inplace=True)

# Drop the first 12 rows of NaNs from the YoY transforms
df = df.iloc[13:, :]

# Scale
scaled_data, scaler = dataset_scaler(df)
scaled_data.columns = df.columns

# ---- Model parameters ----
nvar = scaled_data.shape[1] - 1
n_features = nvar
output_size = 1   # one-step-ahead forecast
n_splits = 5
variable_names = df.columns

cv_dataset = scaled_data[:'2015-12-01']
test_data  = scaled_data['2015-12-01':]

criterion = nn.MSELoss()

# ---- Hyperparameter search ----
# Set CV = True to re-run Bayesian optimization; False to reuse the saved-best config.
CV = False
if CV:
    n_calls = 70
    n_initial_points = 30
    torch.cuda.empty_cache()

    space = [
        Categorical([4, 6, 12, 24], name='lag_num'),
        Real(0.0001, 0.1,           name='learn_rate'),
        Integer(300, 800,           name='num_epochs'),
        Integer(19 * 2, 19 * 10,    name='hidden_size'),
        Integer(1, 7,               name='num_layers'),
        Real(0.0, 0.7,              name='dropout'),
    ]

    LSTM_best_config, LSTM_results_list = bayesian_optimization(
        pd.DataFrame(cv_dataset), LSTM, criterion,
        n_splits, device, space,
        n_calls=n_calls, n_initial_points=n_initial_points,
        n_repeats=1, seed=seed,
    )
    print(f"Best LSTM Configuration: {LSTM_best_config}")

    lag_num, learn_rate, num_epochs, hidden_size, num_layers, drop_out = (
        LSTM_best_config[k] for k in
        ['lag_num', 'learn_rate', 'num_epochs', 'hidden_size', 'num_layers', 'dropout']
    )
else:
    # Best so far - DO NOT DELETE
    lag_num     = 24
    learn_rate  = 0.0001
    num_epochs  = 389
    hidden_size = 224
    num_layers  = 1
    drop_out    = 0.0

# ---- Set up tensors for the rolling-origin forecast ----
cut_off_date = '2016-01-01'   # first test date is 2016-03-01
rolling_df = scaled_data.copy()
cut_off_index = rolling_df.index.get_loc(cut_off_date)
ref_date_series = rolling_df.index[cut_off_index:]
cut_off_index_offset = cut_off_index - lag_num

all_XX, all_YY = XY_gen(rolling_df, lag_num, 1, X_input=[], Y_output=['FF'], overlapping=True)
all_XX, all_YY = torch.tensor(np.array(all_XX)), torch.tensor(np.array(all_YY))


In [ ]:

LSTM_pred_Y = []
VAR_pred_Y = []
Test_Y = []


for i in range(cut_off_index_offset, all_XX.shape[0]):
    if i % 10 == 0:
        print(i)
        
    #3D for LSTM
    train_XX_temp = all_XX[:i,:,:]
    train_YY_temp = all_YY[:i,:,:]
    
    #Only get the next observation
    test_XX_temp = all_XX[i:i+1,:,:]
    test_YY_temp = all_YY[i:i+1,:,:]
    
    #2D for VAR and results
    train_X_temp = train_XX_temp[:,-1,:]
    train_Y_temp = train_YY_temp[:,0,:]
    
    test_X_temp = test_XX_temp[:,-1,:]
    test_Y_temp = test_YY_temp[:,0,:] #also the key reference test results.
    
    #Train and test LSTM model
    
    model_LSTM_temp, _ = Train_DL_Model(LSTM, 
                                        train_XX_temp, train_YY_temp, 
                                        n_features, hidden_size, num_layers, drop_out, 
                                        learn_rate, num_epochs, criterion, device)
    model_LSTM_temp.eval()
    with torch.no_grad():
        LSTM_pred_YY_temp, (_) = model_LSTM_temp(test_XX_temp.float().to(torch.device(device))) #3D
        
    #Transform to 2D array then append
    LSTM_pred_YY_temp = np.array(LSTM_pred_YY_temp.to('cpu'))[:,-1,:]
    LSTM_pred_Y.append(LSTM_pred_YY_temp)
    
    #VAR:
    VAR_model_temp = VAR(np.array(train_X_temp))
    VAR_model_temp = VAR_model_temp.fit(lag_num)
    
    VAR_forecast_temp = VAR_model_temp.forecast(np.array(test_XX_temp[0,:,:]), steps = 1)
    VAR_forecast_temp = VAR_forecast_temp[0]
    
    VAR_pred_Y.append(VAR_forecast_temp)
    
    #Also store the true Y
    Test_Y.append(test_Y_temp)




In [ ]:
RooS_LSTM_pred_Y_df = pd.DataFrame(np.vstack(LSTM_pred_Y)).reset_index()
RooS_VAR_pred_Y_df = pd.DataFrame(np.vstack(VAR_pred_Y)).reset_index()
RooS_Test_Y_df = pd.DataFrame(np.vstack(Test_Y)).reset_index()


RooS_LSTM_pred_Y_df.columns =  ['index',"FF"]

RooS_Test_Y_df.columns =  ['index',"FF"]

RooS_LSTM_pred_Y_df['Label'] = "LSTM, withCOVID"
RooS_Test_Y_df['Label'] = "Actual, withCOVID"

RooS_LSTM_pred_Y_df['Date'] = ref_date_series[1:]
RooS_Test_Y_df['Date'] = ref_date_series[1:]

RooS_Master_res = pd.concat([RooS_LSTM_pred_Y_df, RooS_Test_Y_df])


In [ ]:
import numpy as np

def unscale_target_only(actual_scaled_series, pred_scaled_series, scaled_data, scaler, target_col_index=0):
    """
    Inversely transform scaled target values using the original scaler applied to the full dataset.

    Parameters:
    - actual_scaled_series: pd.Series or np.array of actual target values (scaled)
    - pred_scaled_series: pd.Series or np.array of predicted target values (scaled)
    - scaled_data: np.array or DataFrame that was originally scaled
    - scaler: the fitted scaler (e.g., MinMaxScaler, StandardScaler)
    - target_col_index: index of the target column in the original dataset

    Returns:
    - actual_unscaled: np.array of unscaled actual values
    - pred_unscaled: np.array of unscaled predicted values
    """
    actual_scaled = np.array(actual_scaled_series).reshape(-1, 1)
    pred_scaled = np.array(pred_scaled_series).reshape(-1, 1)

    dummy_actual = np.zeros((len(actual_scaled), scaled_data.shape[1]))
    dummy_pred = np.zeros((len(pred_scaled), scaled_data.shape[1]))

    dummy_actual[:, target_col_index] = actual_scaled[:, 0]
    dummy_pred[:, target_col_index] = pred_scaled[:, 0]

    actual_unscaled = scaler.inverse_transform(dummy_actual)[:, target_col_index]
    pred_unscaled = scaler.inverse_transform(dummy_pred)[:, target_col_index]

    return actual_unscaled, pred_unscaled

actual_unscaled, pred_unscaled = unscale_target_only(
    RooS_Test_Y_df.iloc[:, 1],
    RooS_LSTM_pred_Y_df.iloc[:, 1],
    scaled_data,
    scaler,
    target_col_index=0
)


output_df = pd.DataFrame({
    'Actual_Unscaled': actual_unscaled,
    'Pred_Unscaled': pred_unscaled,
    'Actual_Scaled': RooS_Test_Y_df.iloc[:, 1], 
    'Pred_Scaled': RooS_LSTM_pred_Y_df.iloc[:, 1]
})

from datetime import date, datetime
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save to CSV
output_df.to_csv(f"v3LSTM_unscaled_predictions_{current_time}.csv", index=False)


In [ ]:
# Out-of-sample metrics
rmse = np.sqrt(np.mean((actual_unscaled - pred_unscaled) ** 2))
mae  = np.mean(np.abs(actual_unscaled - pred_unscaled))
bias = np.mean(pred_unscaled - actual_unscaled)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"Bias (pred - actual): {bias:+.4f}")

fig, ax = plt.subplots(1, 1, figsize=(15, 7))
ax.plot(ref_date_series[1:], actual_unscaled, linestyle='--', color='black', label='Actual')
ax.plot(ref_date_series[1:], pred_unscaled,  linestyle='-',  color='blue',
        label='Out-of-sample LSTM predictions')
ax.set_ylabel('Fed Funds Rate')
ax.set_xlabel('Date')
ax.legend(loc='upper left')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(f"v3LSTMroosFed_{current_time}.png", dpi=300)
plt.show()


In [ ]:
# Fixed-sample experiment: train on data up to 2015-12, predict 2016+
_actual_2016, _pred_2016, _dates_2016 = fixed_sample_experiment(
    '2016-01-01', save_tag='2016',
)

In [ ]:
# Fixed-sample experiment: train on data up to 2019-12, predict 2020+
_actual_2020, _pred_2020, _dates_2020 = fixed_sample_experiment(
    '2020-01-01', save_tag='2020',
)